# Módulo 2 — GitHub, CTEs, Window Functions y EDA
**Bootcamp: Fundamentos de Ingeniería de Datos**

En este módulo trabajaremos con un dataset de propiedades inmobiliarias argentinas cargado en una tabla Delta gestionada por Unity Catalog.

---

## Contenido del módulo
1. **EDA** — Exploración y análisis descriptivo del dataset
2. **CTEs** — Common Table Expressions (WITH)
3. **Window Functions** — Funciones de ventana (OVER, PARTITION BY)

---

**Tabla:** `bootcamp.bronze.properties_bronze` (Unity Catalog)
- 1,246,717 filas × 22 columnas
- Fuente: Listings inmobiliarios (zonaprop.com.ar)

## ⚙ SETUP — Verificar tabla bronze en Unity Catalog

El dataset fue cargado en la tabla `bootcamp.bronze.properties_bronze` desde el volume `bootcamp.practice.modulo_2_data`.
Verificamos que la tabla existe y tiene datos antes de comenzar los ejercicios.

In [0]:
# Preparación: Crear schemas y tabla bronze
spark.sql("CREATE SCHEMA IF NOT EXISTS bootcamp.landing")
print("✅ Schema bootcamp.landing creado")

spark.sql("CREATE SCHEMA IF NOT EXISTS bootcamp.bronze")
print("✅ Schema bootcamp.bronze creado")

# Crear tabla bronze desde el CSV en el volume (si no existe)
df = spark.read.csv(
    "/Volumes/bootcamp/practice/modulo_2_data/properties_raw.csv",
    header=True,
    inferSchema=True
)
df.write.saveAsTable("bootcamp.bronze.properties_bronze", mode="overwrite")

# Verificar que la tabla existe y tiene datos
count = spark.sql("SELECT COUNT(*) AS total FROM bootcamp.bronze.properties_bronze").collect()[0]['total']
print(f"✅ Tabla bootcamp.bronze.properties_bronze verificada — {count:,} filas")

## PARTE 1: EDA (Exploratory Data Analysis)

Explorar el dataset para entender su estructura, calidad de datos y distribuciones.

**Instrucciones:** Resolvé cada ejercicio en la celda correspondiente.

In [0]:
%sql
-- Ejercicio 1.1: ¿Cuántas filas y columnas tiene el dataset?
-- Usá DESCRIBE para ver los tipos de datos de cada columna.
-- ¿Qué columnas están como STRING que probablemente deberían ser numéricas?

DESCRIBE bootcamp.bronze.properties_bronze;

-- 1) El dataset tiene 22 columnas y 1.246.717 filas.
-- 2) Las columnas que probablemente deberían ser numéricas son: precio, numero, expensas, ambientes, metros_cuadrados_totales, metros_cuadrados_cubiertos, piso, antiguedad.

In [0]:
%sql
SELECT COUNT(*)
FROM bootcamp.bronze.properties_bronze;

In [0]:
%sql
-- Ejercicio 1.2: Contá cuántos valores NULL tiene cada columna.
-- Ordená de mayor a menor cantidad de nulos.
-- Reflexión: ¿Qué columnas tienen más datos faltantes? ¿Por qué podría ser?

SELECT 
    columna,
    SUM(CASE WHEN valor IS NULL OR valor = '' THEN 1 ELSE 0 END) AS total_nulos
FROM (
    SELECT 
        STACK(22,
            'id', CAST(id AS STRING),
            'ubicacion', CAST(ubicacion AS STRING),
            'precio', CAST(precio AS STRING),
            'numero', CAST(numero AS STRING),
            'calle', CAST(calle AS STRING),
            'expensas', CAST(expensas AS STRING),
            'tipo_de_operacion', CAST(tipo_de_operacion AS STRING),
            'moneda', CAST(moneda AS STRING),
            'ambientes', CAST(ambientes AS STRING),
            'metros_cuadrados_totales', CAST(metros_cuadrados_totales AS STRING),
            'metros_cuadrados_cubiertos', CAST(metros_cuadrados_cubiertos AS STRING),
            'orientacion_cardinal', CAST(orientacion_cardinal AS STRING),
            'orientacion_inmueble', CAST(orientacion_inmueble AS STRING),
            'piso', CAST(piso AS STRING),
            'cochera', CAST(cochera AS STRING),
            'antiguedad', CAST(antiguedad AS STRING),
            'estado', CAST(estado AS STRING),
            'tipo_vendedor', CAST(tipo_vendedor AS STRING),
            'url', CAST(url AS STRING),
            'zona', CAST(zona AS STRING),
            'fecha', CAST(fecha AS STRING),
            'hora', CAST(hora AS STRING)
        ) AS (columna, valor)
    FROM bootcamp.bronze.properties_bronze
) t
GROUP BY columna
ORDER BY total_nulos DESC;

-- Usamos STACK para transformar las columnas en filas y ordenarlas de mayor a menor nulos.

-- REFLEXION:
-- 1) Causas raíz desde la Ingeniería de Datos y la Ingesta
--      Diseño deficiente de formularios en origen (UI/UX): No se aplican validaciones (null constraints o campos obligatorios) para metadatos secundarios. Se guardan y publicar avisos incompletos.
--      Los sistemas exigen lo básico para que el aviso sea visible (precio, ubicación general, tipo de operación y al menos una foto o título). Otros detalles, no bloquean la publicación si se omiten, generando una enorme cantidad de nulos.
-- 2) Algunos campos dependen enteramente de la tipología del inmueble.
--      a) piso: Si la propiedad es una casa, un lote o un PH en planta baja sin altura, el campo carece de sentido. Se deja vacíos o nulos.
--      b) cochera: Si el inmueble no posee cochera, muchos avisos simplemente dejan el campo en blanco o nulo.
--      c) expensas: Las casas, terrenos y locales comerciales por lo general no pagan expensas de edificio.
-- 3) Privacidad.
--      a) calle: algunos propietarios o inmobiliarias se niegan a publicar la calle o altura exacta de la propiedad por motivos de privacidad o seguridad. Solo muestran el barrio, la zona o intersección aproximada.


In [0]:
%sql
-- Ejercicio 1.3: ¿Cuántas propiedades hay por zona?
-- Mostrá las 15 zonas con más propiedades.

SELECT 
    zona,
    COUNT(*) AS total_propiedades
FROM 
    bootcamp.bronze.properties_bronze
WHERE 
    zona IS NOT NULL 
    AND zona != ''
GROUP BY 
    zona
ORDER BY 
    total_propiedades DESC
LIMIT 15;

-- Al hacer un GROUP BY sobre una columna de texto en la capa Bronze, filtrar primero los nulos o strings vacíos reduce el volumen de datos intermedios que se mueven por la red, optimizando la ejecución de la consulta.

In [0]:
%sql
-- Ejercicio 1.4: ¿Cuántas propiedades hay por tipo de operación (venta/alquiler) y moneda (USD/ARS)?

SELECT 
    tipo_de_operacion,
    moneda,
    COUNT(*) AS total_propiedades
FROM 
    bootcamp.bronze.properties_bronze
WHERE 
    tipo_de_operacion IS NOT NULL 
    AND tipo_de_operacion != ''
    AND moneda IS NOT NULL 
    AND moneda != ''
GROUP BY 
    tipo_de_operacion,
    moneda
ORDER BY 
    total_propiedades DESC;

-- INFERENCIA PERSONAL sobre la calidad de los datos:
--  1. Desplazamiento de columnas: nombres de calles (Las Malvinas, Salguero, Lacroze, Emilio Lamarca) y precios o números (50000.0, 120000.0), aparecen en las columnas tipo_de_operacion y moneda.
--  2. Falta de estandarización y valores sucios: Incluso en los registros donde los campos sí corresponden a lo esperado, la variabilidad tipográfica es enorme. Múltiples formas de expresar el mismo concepto:
--      tipo_de_operacion: Conviven alquiler_temporal, alquiler_temporario, alquiler_de_temporada, venta/alquiler, venta_y_alquiler y preventa.
--  3. Errores tipográficos humanos: faltas de ortografía o catálogos abiertos como alquier y alquieler.
--  4. Anomalías monetarias y monedas exóticas o erróneas: el grueso del mercado opera en USD y ARS, pero el dataset crudo mezcla valores extraños como MXN (Pesos mexicanos), guaranies o uyu (Pesos uruguayos).

Valores numéricos o strings vacíos haciendo las veces de moneda.

In [0]:
%sql
-- Ejercicio 1.5: Calculá el precio mínimo, máximo y promedio por moneda.
-- Nota: la columna precio está como STRING — usá CAST(precio AS DOUBLE)
-- Reflexión: ¿Los valores extremos tienen sentido o hay outliers?

SELECT
    moneda,
    tipo_de_operacion,
    MIN(TRY_CAST(precio AS DOUBLE)) AS precio_min,
    MAX(TRY_CAST(precio AS DOUBLE)) AS precio_max,
    AVG(TRY_CAST(precio AS DOUBLE)) AS precio_promedio,
    COUNT(*) AS total,
    -- Percentil 50 (Mediana)
    percentile_approx(TRY_CAST(precio AS DOUBLE), 0.5) AS precio_mediana,
    -- Percentil 25 (Q1)
    percentile_approx(TRY_CAST(precio AS DOUBLE), 0.25) AS percentil_25,
    -- Percentil 75 (Q3)
    percentile_approx(TRY_CAST(precio AS DOUBLE), 0.75) AS percentil_75
FROM
    bootcamp.bronze.properties_bronze
WHERE
    moneda IN ('USD', 'ARS')
-- Filtramos solo monedas legítimas para evitar ruido de columnas desplazadas
    AND tipo_de_operacion IN ('venta', 'alquiler', 'alquiler_temporal', 'alquiler_temporario')
-- Tipos de operaciones válidas
    AND TRY_CAST(precio AS DOUBLE) IS NOT NULL
    AND TRY_CAST(precio AS DOUBLE) > 0
GROUP BY
    moneda,
    tipo_de_operacion
ORDER BY
    moneda,
    total DESC;

-- Sí, hay outliers extremos absurdos (Errores de carga o del scraping).

-- El MIN de 0.06 o 1 USD/ARS: En categorías como USD venta hay un precio MIN de USD 0.06. En ARS alquiler o USD alquiler hay propiedades a ARS 1 o USD 1/mes. Podría ser un error de carga, un valor simbólico o esas publicaciónes de "CONSULTAR".
-- El MAX de 1,111,111,111 USD en Venta: Probablemente un error de tipeo o un campo donde se filtró otro tipo de información de otra columna.
-- El MAX de ARS 1,200,000,000 en Alquiler: Idem que el anterior.

-- Revelación:
--  La Mediana frente al AVG.
--      En USD venta:
--              AVG precio da USD 219,097   vs.   La mediana es de USD 135,000.
--      El AVG está sesgado hacia arriba por las pocas propiedades multimillonarias de lujo (mansiones, penthouses). La mediana está infiriendo, con algo más de precisión, cuánto cuesta realmente una propiedad típica o promedio en el mercado analizado.

--      El caso disparatado de ARS alquiler:
--          hay un AVG precio de ARS 793,701, pero con un MAX de ARS 1.200 millones.
--      EsO demuestra cómo un solo outlier extremo puede inflar artificialmente la media aritmética. La mediana (ARS 650,000) se ve algo más robusta y podría estar reflejando el valor central real de los alquileres en pesos.

## PARTE 2: CTEs (Common Table Expressions)

Una CTE es una consulta temporal con nombre que se define con `WITH` y se puede usar dentro de una consulta más grande.
Ventajas: legibilidad, reutilización, recursividad.

```sql
WITH nombre_cte AS (
    SELECT ...
)
SELECT * FROM nombre_cte;
```

In [0]:
%sql
-- Ejercicio 2.1: Usá una CTE para calcular el precio promedio en USD por zona,
-- y luego filtrá solo las zonas con más de 1000 propiedades.

WITH estadisticas_zona AS (
    SELECT 
        zona,
        ROUND(AVG(TRY_CAST(precio AS DOUBLE)),2) AS precio_promedio_usd,
        COUNT(*) AS total_propiedades
    FROM 
        bootcamp.bronze.properties_bronze
    WHERE 
        moneda = 'USD'
        AND zona IS NOT NULL 
        AND zona != ''
        AND TRY_CAST(precio AS DOUBLE) IS NOT NULL
        AND TRY_CAST(precio AS DOUBLE) > 0
    GROUP BY 
        zona
)
SELECT 
    zona,
    precio_promedio_usd,
    total_propiedades
FROM 
    estadisticas_zona
WHERE 
    total_propiedades > 1000
ORDER BY 
    precio_promedio_usd DESC;

In [0]:
%sql
-- Ejercicio 2.2: Usá múltiples CTEs para calcular el precio por metro cuadrado.
-- CTE 1: Filtrar propiedades en USD con metros válidos.
-- CTE 2: Calcular precio_m2 = precio / metros_cuadrados_cubiertos.
-- Query final: Top 10 zonas con mayor precio_m2 promedio.

-- CTE 1: Filtrar propiedades en USD con precios y metros válidos
WITH propiedades_validas AS (
    SELECT
        zona,
        TRY_CAST(precio AS DOUBLE) AS precio,
        TRY_CAST(metros_cuadrados_cubiertos AS DOUBLE) AS metros
    FROM
        bootcamp.bronze.properties_bronze
    WHERE
        moneda = 'USD'
        AND zona IS NOT NULL
        AND zona != ''
        AND TRY_CAST(precio AS DOUBLE) IS NOT NULL
        AND TRY_CAST(precio AS DOUBLE) > 0
        AND TRY_CAST(metros_cuadrados_cubiertos AS DOUBLE) IS NOT NULL
        AND TRY_CAST(metros_cuadrados_cubiertos AS DOUBLE) > 0
),

-- CTE 2: Calcular el precio por metro cuadrado individual por propiedad
precio_m2 AS (
    SELECT
        zona,
        precio / metros AS precio_por_m2,
        metros
    FROM propiedades_validas
)

-- Query final: Top 10 zonas con mayor precio_m2 promedio
SELECT
    zona,
    ROUND(AVG(precio_por_m2),2) AS precio_m2_promedio,
    COUNT(*) AS total
FROM precio_m2
GROUP BY zona
HAVING COUNT(*) > 50
ORDER BY precio_m2_promedio DESC
LIMIT 10;

In [0]:
%sql
-- Ejercicio 2.3: Usá CTEs para comparar el precio promedio de alquiler vs venta por zona.
-- Mostrá zonas donde hay al menos 50 propiedades de cada tipo.

WITH alquileres AS (
    SELECT
        zona,
        ROUND(AVG(CAST(precio AS DOUBLE)),2) AS precio_alquiler,
        COUNT(*) AS total
    FROM
        bootcamp.bronze.properties_bronze
    WHERE
        tipo_de_operacion = 'alquiler'
        AND precio IS NOT NULL
    GROUP BY
        zona
),
ventas AS (
    SELECT
        zona,
        ROUND(AVG(CAST(precio AS DOUBLE)),2) AS precio_venta,
        COUNT(*) AS total
    FROM
        bootcamp.bronze.properties_bronze
    WHERE
        tipo_de_operacion = 'venta'
        AND precio IS NOT NULL
    GROUP BY
        zona
)
SELECT
    a.zona,
    a.precio_alquiler,
    v.precio_venta,
    v.precio_venta / a.precio_alquiler AS ratio_venta_alquiler,
    a.total AS total_alquileres,
    v.total AS total_ventas
FROM alquileres a
JOIN ventas v ON a.zona = v.zona
WHERE a.total >= 50 AND v.total >= 50
ORDER BY ratio_venta_alquiler DESC;

## PARTE 3: Window Functions

Las window functions realizan cálculos sobre un conjunto de filas relacionadas (una "ventana") sin colapsarlas en una sola fila.

```sql
funcion() OVER (
    PARTITION BY columna  -- agrupar
    ORDER BY columna      -- ordenar dentro del grupo
)
```

Funciones comunes: `ROW_NUMBER()`, `RANK()`, `DENSE_RANK()`, `LAG()`, `LEAD()`, `SUM() OVER`, `AVG() OVER`

In [0]:
%sql
-- Ejercicio 3.1: Usá ROW_NUMBER() para encontrar la propiedad más cara de cada zona (en USD).
-- Mostrá zona, precio, ubicacion y ambientes de esas propiedades.

WITH propiedades_usd AS (
    SELECT
        zona,
        ubicacion,
        CAST(precio AS DOUBLE) AS precio_num,
        ambientes,
        ROW_NUMBER() OVER (
            PARTITION BY zona
            ORDER BY CAST(precio AS DOUBLE) DESC
        ) AS rn
    FROM bootcamp.bronze.properties_bronze
    WHERE moneda = 'USD' AND precio IS NOT NULL
)
SELECT zona, ubicacion, precio_num AS precio, ambientes
FROM propiedades_usd
WHERE rn = 1
ORDER BY precio_num DESC
LIMIT 20;

In [0]:
%sql
-- Ejercicio 3.2: Para la zona 'palermo', rankeá las propiedades por precio usando RANK() y DENSE_RANK().
-- ¿Cuál es la diferencia entre RANK y DENSE_RANK cuando hay empates?

SELECT
    ubicacion,
    CAST(precio AS DOUBLE) AS precio,
    RANK() OVER (ORDER BY CAST(precio AS DOUBLE) DESC) AS ranking_rank,
    DENSE_RANK() OVER (ORDER BY CAST(precio AS DOUBLE) DESC) AS ranking_dense
FROM bootcamp.bronze.properties_bronze
WHERE zona = 'palermo'
  AND moneda = 'USD'
  AND precio IS NOT NULL
ORDER BY precio DESC
LIMIT 20;

In [0]:
%sql
-- Ejercicio 3.3: Usá LAG() y LEAD() para comparar el precio de cada propiedad con la anterior y la siguiente.
-- Filtrá por zona 'belgrano', moneda USD, y ordená por precio.
-- Calculá la diferencia de precio con la propiedad anterior.

SELECT
    ubicacion,
    CAST(precio AS DOUBLE) AS precio,
    LAG(CAST(precio AS DOUBLE)) OVER (ORDER BY CAST(precio AS DOUBLE)) AS precio_anterior,
    LEAD(CAST(precio AS DOUBLE)) OVER (ORDER BY CAST(precio AS DOUBLE)) AS precio_siguiente,
    CAST(precio AS DOUBLE) - LAG(CAST(precio AS DOUBLE)) OVER (ORDER BY CAST(precio AS DOUBLE)) AS diferencia_anterior
FROM bootcamp.bronze.properties_bronze
WHERE zona = 'belgrano'
  AND moneda = 'USD'
  AND precio IS NOT NULL
ORDER BY precio DESC
LIMIT 20;

In [0]:
%sql
-- Ejercicio 3.4: Calculá el promedio acumulado de precios por zona a medida que se recorren las propiedades.
-- Usá AVG() OVER con PARTITION BY zona y ORDER BY precio.

SELECT
    zona,
    ubicacion,
    CAST(precio AS DOUBLE) AS precio,
    AVG(CAST(precio AS DOUBLE)) OVER (
        PARTITION BY zona
        ORDER BY CAST(precio AS DOUBLE)
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS promedio_acumulado
FROM bootcamp.bronze.properties_bronze
WHERE moneda = 'USD'
  AND precio IS NOT NULL
  AND zona IN ('palermo', 'belgrano', 'recoleta')
ORDER BY zona, precio DESC
LIMIT 30;

In [0]:
%sql
-- Ejercicio 3.5: Calculá el porcentaje que representa cada propiedad respecto al total de precios de su zona.
-- Usá SUM() OVER (PARTITION BY zona) para obtener el total por zona.

SELECT
    zona,
    ubicacion,
    CAST(precio AS DOUBLE) AS precio,
    SUM(CAST(precio AS DOUBLE)) OVER (PARTITION BY zona) AS total_zona,
    ROUND(
        (CAST(precio AS DOUBLE) / SUM(CAST(precio AS DOUBLE)) OVER (PARTITION BY zona)) * 100,
        4
    ) AS porcentaje_del_total
FROM bootcamp.bronze.properties_bronze
WHERE moneda = 'USD'
  AND precio IS NOT NULL
  AND zona IN ('palermo', 'caballito', 'flores')
ORDER BY zona, porcentaje_del_total DESC
LIMIT 30;